#  Text Feature Extraction

In [1]:
import os
import numpy as np
import pandas as pd

In [ ]:
dataset_name = 'Beauty'
os.chdir('./datasets/Amazon-2014/%s/5-core'%dataset_name)
os.getcwd()

## Load meta data

In [ ]:
i_id, desc_str = 'itemID', 'description'

file_path = './'
file_name = 'meta-%s.csv'%dataset_name

meta_file = os.path.join(file_path, file_name)

df = pd.read_csv(meta_file)
df.sort_values(by=[i_id], inplace=True)

print('data loaded!')
print(f'shape: {df.shape}')

df[:3]

In [ ]:
# sentences: title + brand + category + description | All have title + description
title_na_df = df[df['title'].isnull()]
print(title_na_df.shape)

desc_na_df = df[df['description'].isnull()]
print(desc_na_df.shape)

na_df = df[df['description'].isnull() & df['title'].isnull()]
print(na_df.shape)

na3_df = df[df['description'].isnull() & df['title'].isnull() & df['brand'].isnull()]
print(na3_df.shape)

na4_df = df[df['description'].isnull() & df['title'].isnull() & df['brand'].isnull() & df['categories'].isnull()]
print(na4_df.shape)

In [ ]:
df[desc_str] = df[desc_str].fillna(" ")
df['title'] = df['title'].fillna(" ")
df['brand'] = df['brand'].fillna(" ")
df['categories'] = df['categories'].fillna(" ")
print(f'columns: {df.columns}')
print(df.head())

In [ ]:
import re

def text_cleaning(input_text):
    pattern = re.compile(r'\.caption\s*{.*?}|\w+\.(\w+)\s*{[^}]*}|- <.*?> -|\[if gte mso.*?\[endif\]', re.DOTALL)
    cleaned_text = re.sub(pattern, '', input_text)

    return cleaned_text

In [ ]:
import html

sentences = []
item_ids = []
for i, row in df.iterrows():
    if row['title'] == ' ' and row['brand'] == ' ' and row['categories'] == ' ' and row[desc_str] == ' ':
        continue
    else:
        sen = row['title'] + ' ' + row['brand'] + ' '
        categories=eval(row['categories'])
        if isinstance(categories,list):
            for c in categories[0]:
                sen=sen+c+" "
        sen += text_cleaning(row[desc_str])
        sen = sen.replace('\n', ' ')

        sen = html.unescape(sen)
        sentences.append(sen)
        item_ids.append(row['itemID'])

sentences[:10]

In [8]:
course_list = df[i_id].tolist()
#sentences = df[desc_str].tolist()

assert course_list[-1] == len(course_list) - 1

In [9]:
import math
from tqdm import tqdm

def extract_raw_text_feature(text_corpus, tokenizer, model, batch_size=512):
    batch_num = math.ceil(len(text_corpus)/batch_size)
    features = []
    for batch_idx in tqdm(range(batch_num)):
        processing_text = sentences[batch_idx*batch_size: (batch_idx+1)*batch_size]
        inputs = tokenizer(processing_text, return_tensors="pt", padding=True, truncation=True, max_length=512)  # Batch size 1
        inputs = inputs.to(model.device)
        attention_mask = inputs['attention_mask']
        input_ids = inputs.input_ids
        encoded_sentence = model(input_ids, attention_mask)
        last_hidden_states = encoded_sentence.last_hidden_state
        sentence_embedding = last_hidden_states.mean(dim=1)

        if len(features) == 0:
            features = sentence_embedding.detach().cpu().numpy()
        else:
            features = np.concatenate((features, sentence_embedding.detach().cpu().numpy()), axis=0)
    
    return features

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

gpu_id = 0
device = torch.device('cuda:' + str(gpu_id) if torch.cuda.is_available() else 'cpu')

tokenizer = AutoTokenizer.from_pretrained('./pretrain_models/xlm-roberta-base')
model = AutoModel.from_pretrained('./pretrain_models/xlm-roberta-base')
model = model.to(device)

batch_size = 4
features = extract_raw_text_feature(sentences, tokenizer, model, batch_size)

print(features.shape)
features[:5]

In [ ]:
average_feature = np.mean(features, axis=0)

features_array = np.empty((df.shape[0], features.shape[-1]))
idx = 0
for item_id in df['itemID'].values:
    if item_id not in item_ids:
        features_array[item_id] = average_feature  
    else:
        features_array[item_id] = features[idx]  
        idx += 1

assert features_array.shape[0] == df.shape[0]

print(features_array.shape)
features_array[:5]

In [ ]:
#save
np.save(os.path.join(file_path, 'text_feat.npy'), features_array)
print('done!')

In [ ]:
# load
load_txt_feat = np.load('text_feat.npy', allow_pickle=True)
print(load_txt_feat.shape)
load_txt_feat[:10]